In [ ]:
from private_config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#初版复现的GPN,粗糙复现，效果差
#怀疑作者在论文中将ResNeXt的模块绘制出错：块的最后一个有效卷积层应为<configured>*<configured>卷积而不是<configured>*<configured>组卷积
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from data_loader import *

class SEModule(nn.Module):
    """Squeeze-and-Excitation模块"""
    def __init__(self, channels, reduction=None):
        reduction = _cfg_resolve('old/GPN_old.ipynb.cell0.SEModule.__init__.reduction', reduction)
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

# class ResNeXtBottleneck(nn.Module):
#     """ResNeXt瓶颈块，包含组卷积和SE模块"""
# Historical configuration omitted; provide values through private configuration.
#         super(ResNeXtBottleneck, self).__init__()
        
#         # 计算组卷积的中间通道数
# Historical configuration omitted; provide values through private configuration.
        
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
        
#         # 组卷积
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
        
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
        
#         # SE模块
#         self.se = SEModule(out_channels)
        
#         self.relu = nn.ReLU(inplace=True)
        
#         # 残差连接
#         self.shortcut = nn.Sequential()
# Historical configuration omitted; provide values through private configuration.
#             self.shortcut = nn.Sequential(
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
#             )

#     def forward(self, x):
#         residual = self.shortcut(x)
        
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
#         out = self.relu(out)
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
#         out = self.relu(out)
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
#         # SE模块
#         out = self.se(out)
        
#         out += residual
#         out = self.relu(out)
        
#         return out

class BottleneckBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=None):
        stride = _cfg_resolve('old/GPN_old.ipynb.cell0.BottleneckBlock.__init__.stride', stride)
        super(BottleneckBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels // 4, kernel_size=_cfg_require('old/GPN_old.ipynb.cell0.BottleneckBlock.__init__.kernel_size'), bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels // 4)
        self.conv2 = nn.Conv2d(out_channels // 4, out_channels // 4, kernel_size=_cfg_require('old/GPN_old.ipynb.cell0.BottleneckBlock.__init__.kernel_size__2'), stride=stride, padding=_cfg_require('old/GPN_old.ipynb.cell0.BottleneckBlock.__init__.padding'), bias=False, groups=out_channels // 4)
        self.bn2 = nn.BatchNorm2d(out_channels // 4)
        self.conv3 = nn.Conv2d(out_channels // 4, out_channels, kernel_size=_cfg_require('old/GPN_old.ipynb.cell0.BottleneckBlock.__init__.kernel_size__3'),padding=_cfg_require('old/GPN_old.ipynb.cell0.BottleneckBlock.__init__.padding__2'),bias=False, groups=out_channels // 4)        
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_channels, out_channels // 16, kernel_size=_cfg_require('old/GPN_old.ipynb.cell0.BottleneckBlock.__init__.kernel_size__4')),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels // 16, out_channels, kernel_size=_cfg_require('old/GPN_old.ipynb.cell0.BottleneckBlock.__init__.kernel_size__5')),
            nn.Sigmoid()
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=_cfg_require('old/GPN_old.ipynb.cell0.BottleneckBlock.__init__.kernel_size__6'), stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)
        out = self.conv3(out)
        out = self.bn3(out)
        se_out = self.se(out)
        out = out * se_out
        out += self.shortcut(x)
        out = self.relu(out)
        return out
class GPN(nn.Module):
    """
    高斯原型网络 (GPN) - 基于论文实现
    输出embedding feature v和precision feature s
    """
    def __init__(self,feature_dim=None, groups=None, base_width=None):
        feature_dim = _cfg_resolve('old/GPN_old.ipynb.cell0.GPN.__init__.feature_dim', feature_dim)
        groups = _cfg_resolve('old/GPN_old.ipynb.cell0.GPN.__init__.groups', groups)
        base_width = _cfg_resolve('old/GPN_old.ipynb.cell0.GPN.__init__.base_width', base_width)
        super().__init__()
        
        self.feature_dim = feature_dim
        # 初始卷积层
        self.conv1 = nn.Conv2d(_cfg_require('old/GPN_old.ipynb.cell0.GPN.__init__.Conv2d_arg0'), int(feature_dim/8), kernel_size=_cfg_require('old/GPN_old.ipynb.cell0.GPN.__init__.kernel_size'), stride=_cfg_require('old/GPN_old.ipynb.cell0.GPN.__init__.stride'), padding=_cfg_require('old/GPN_old.ipynb.cell0.GPN.__init__.padding'), bias=False)
        self.bn1 = nn.BatchNorm2d(int(feature_dim/8))
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=_cfg_require('old/GPN_old.ipynb.cell0.GPN.__init__.kernel_size__2'), stride=_cfg_require('old/GPN_old.ipynb.cell0.GPN.__init__.stride__2'), padding=_cfg_require('old/GPN_old.ipynb.cell0.GPN.__init__.padding__2'))

        self.layer1 = self._make_block(int(feature_dim/8), int(feature_dim/8),stride=_cfg_require('old/GPN_old.ipynb.cell0.GPN.__init__.stride__3'))
        self.layer2 = self._make_block(int(feature_dim/8), int(feature_dim/4), stride=_cfg_require('old/GPN_old.ipynb.cell0.GPN.__init__.stride__4'))
        self.layer3 = self._make_block(int(feature_dim/4), int(feature_dim/2), stride=_cfg_require('old/GPN_old.ipynb.cell0.GPN.__init__.stride__5'))
        self.layer4 = self._make_block(int(feature_dim/2), feature_dim, stride=_cfg_require('old/GPN_old.ipynb.cell0.GPN.__init__.stride__6'))
        # Channel Half操作 - 分成两个通道
        self.channel_half_v = nn.Conv2d(feature_dim, int(feature_dim/2), kernel_size=_cfg_require('old/GPN_old.ipynb.cell0.GPN.__init__.kernel_size__3'), bias=False)
        self.channel_half_s = nn.Conv2d(feature_dim, int(feature_dim/2), kernel_size=_cfg_require('old/GPN_old.ipynb.cell0.GPN.__init__.kernel_size__4'), bias=False)
        
        # 全局平均池化
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        self._initialize_weights()

    def _make_block(self, in_channels, out_channels, stride):
        layers = []
        layers.append(BottleneckBlock(in_channels, out_channels, stride))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        # 输入: [batch_size, <configured>, <configured>, <configured>]
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)  # [batch_size, <configured>, <configured>, <configured>]

        # Channel Half操作
        v_features = self.channel_half_v(x)  # [batch_size, feature_dim, <configured>, <configured>]
        s_features = self.channel_half_s(x)  # [batch_size, feature_dim, <configured>, <configured>]
        
        # 全局平均池化
        v = self.avgpool(v_features).flatten(1)  # [batch_size, feature_dim]
        s = self.avgpool(s_features).flatten(1)  # [batch_size, feature_dim]
        
        return v, s

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

class GPNLoss(nn.Module):
    """
    GPN损失函数 - 马氏距离 + 交叉熵
    """
    def __init__(self):
        super().__init__()
        
    def compute_mahalanobis_distance(self, v, prototypes, precision_matrices):
        """
        计算马氏距离
        
        Args:
            v: 查询样本特征 [batch_size, feature_dim]
            prototypes: 各类原型 [n_ways, feature_dim]
            precision_matrices: 各类精度矩阵 [n_ways, feature_dim, feature_dim]
            
        Returns:
            distances: 马氏距离 [batch_size, n_ways]
        """
        batch_size, feature_dim = v.shape
        n_ways = prototypes.shape[0]

        # 扩维以便广播计算
        v_expanded = v.unsqueeze(1).expand(-1, n_ways, -1)  # [batch_size, n_ways, feature_dim]
        prototypes_expanded = prototypes.unsqueeze(0).expand(batch_size, -1, -1)  # [batch_size, n_ways, feature_dim]

        # 计算差异
        diff = v_expanded - prototypes_expanded  # [batch_size, n_ways, feature_dim]

        distances = torch.zeros(batch_size, n_ways, device=v.device)

        for i in range(n_ways):
            diff_i = diff[:, i, :]  # [batch_size, feature_dim]
            P_i = precision_matrices[i]  # [feature_dim, feature_dim]

            # 计算二次型: (x-μ)^T P (x-μ)
            # 使用批矩阵乘法提高效率
            temp = torch.mm(diff_i, P_i)  # [batch_size, feature_dim]
            quadratic_form = torch.sum(diff_i * temp, dim=1)  # [batch_size]
            distances[:, i] = torch.sqrt(torch.clamp(quadratic_form, min=1e-8))

        return distances
    
    def forward(self, query_v, prototypes, precision_matrices, query_labels):
        """
        计算损失
        
        Args:
            query_v: 查询样本特征 [batch_size, feature_dim]
            prototypes: 各类原型 [n_ways, feature_dim]
            precision_matrices: 各类精度矩阵 [n_ways, feature_dim, feature_dim]
            query_labels: 查询集标签 [batch_size]
        """
        # 计算马氏距离
        distances = self.compute_mahalanobis_distance(query_v, prototypes, precision_matrices)

        # 将距离转换为logits（负距离）
        logits = -distances

        # 计算softmax概率
        probabilities = F.softmax(logits, dim=1)

        # 计算交叉熵损失
        loss = F.cross_entropy(logits, query_labels)

        return loss, probabilities, distances

class GPNTrainer:
    """
    GPN模型训练器
    """
    def __init__(self, model,device,feature_dim=None, lr=None):
        feature_dim = _cfg_resolve('old/GPN_old.ipynb.cell0.GPNTrainer.__init__.feature_dim', feature_dim)
        lr = _cfg_resolve('old/GPN_old.ipynb.cell0.GPNTrainer.__init__.lr', lr)
        self.device = device
        self.feature_dim = feature_dim
        
        # 初始化模型
        self.model = model
        self.model.to(device)
        self.loss_fn = GPNLoss()
        self.lr=lr
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)


        print(f"GPN模型初始化完成，设备: {self.device}")
        print(f"参数量: {sum(p.numel() for p in self.model.parameters()):,}")

    def compute_gaussian_prototypes(self, support_v, support_s, support_labels, n_ways):
        '\n        计算高斯原型和精度矩阵（论文公式<configured>-<configured>）\n        \n        Args:\n            support_v: 支持集embedding特征 [n_ways * k_shot, feature_dim]\n            support_s: 支持集precision特征 [n_ways * k_shot, feature_dim]\n            support_labels: 支持集标签 [n_ways * k_shot]\n            n_ways: 类别数\n            k_shot: 每类样本数\n            \n        Returns:\n            prototypes: 各类原型 [n_ways, feature_dim]\n            precision_matrices: 各类精度矩阵 [n_ways, feature_dim, feature_dim]\n        '
        feature_dim = support_v.shape[1]

        prototypes = torch.zeros(n_ways, feature_dim, device=self.device)
        precision_matrices = torch.zeros(n_ways, feature_dim, feature_dim, device=self.device)

        # Historical configuration omitted; provide values through private configuration.
        sigma = support_s

        # 为每个类别计算原型
        unique_labels = torch.unique(support_labels)
        
        for i, class_label in enumerate(unique_labels):
            class_mask = (support_labels == class_label)
            
            class_v = support_v[class_mask]  # [k_shot, feature_dim]
            class_sigma = sigma[class_mask]  # [k_shot, feature_dim]
            
            # Historical configuration omitted; provide values through private configuration.
            weighted_sum = torch.sum(class_sigma * class_v, dim=0)  # [feature_dim]
            sigma_sum = torch.sum(class_sigma, dim=0)  # [feature_dim]
            prototypes[i] = weighted_sum / (sigma_sum + 1e-8)

            # 计算精度矩阵（对角矩阵，公式<configured>）
            precision_diag = torch.mean(class_sigma, dim=0)  # [feature_dim]
            precision_matrices[i] = torch.diag(precision_diag)

        return prototypes, precision_matrices

    # def train_step(self, support_signals, support_labels, query_signals, query_labels,n_ways):
    #     """
    #     单次训练步骤
    #     """
    #     self.model.train()

    #     # 移动到设备
    #     support_signals = support_signals.to(self.device).float()
    #     support_labels = support_labels.to(self.device)
    #     query_signals = query_signals.to(self.device).float()
    #     query_labels = query_labels.to(self.device)

    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    #     # 前向传播
    #     support_v, support_s = self.model(support_signals)
    #     query_v, query_s = self.model(query_signals)

    #     # 重新映射标签到<configured>, <configured>, <configured>, ..., n_ways-<configured>
    #     unique_labels = torch.unique(support_labels)
    #     label_mapping = {label.item(): i for i, label in enumerate(unique_labels)}
        
    #     remapped_support_labels = torch.tensor([label_mapping[label.item()] for label in support_labels], 
    # Historical configuration omitted; provide values through private configuration.
    #     remapped_query_labels = torch.tensor([label_mapping[label.item()] for label in query_labels], 
    # Historical configuration omitted; provide values through private configuration.

    #     # 计算高斯原型
    #     prototypes, precision_matrices = self.compute_gaussian_prototypes(
    #         support_v, support_s, remapped_support_labels, n_ways
    #     )

    #     # 计算损失
    #     loss, probabilities, distances = self.loss_fn(
    #         query_v, prototypes, precision_matrices, remapped_query_labels
    #     )

    #     # 反向传播
    #     self.optimizer.zero_grad()
    #     loss.backward()
        
    #     # 梯度裁剪
    # Historical configuration omitted; provide values through private configuration.
        
    #     self.optimizer.step()

    #     # 计算准确率
    # Historical configuration omitted; provide values through private configuration.
    #     accuracy = (predictions == remapped_query_labels).float().mean()

    #     return loss.item(), accuracy.item()
    def train_step_batch(self, meta_batch):
        """
        批处理训练步骤 - 累积梯度后统一更新
        """
        self.model.train()
        
        batch_size = len(meta_batch[0])  # meta_batch中任务的数量
        total_loss = 0
        total_acc = 0
        
        # 清零梯度
        self.optimizer.zero_grad()
        
        # 处理批内所有任务，累积梯度
        for i in range(batch_size):
            support_signals, support_labels = meta_batch[0][i], meta_batch[1][i]
            query_signals, query_labels = meta_batch[2][i], meta_batch[3][i]
            
            # 单个任务的前向传播和损失计算
            loss, acc = self._single_task_forward(
                support_signals, support_labels, 
                query_signals, query_labels
            )
            
            # 累积损失（自动累积梯度）
            (loss / batch_size).backward()  # 除以batch_size来平均化梯度
            
            total_loss += loss.item()
            total_acc += acc
        
        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=_cfg_require('old/GPN_old.ipynb.cell0.GPNTrainer.train_step_batch.max_norm'))
        
        # 统一更新参数
        self.optimizer.step()
        
        return total_loss / batch_size, total_acc / batch_size

    def _single_task_forward(self, support_signals, support_labels, query_signals, query_labels):
        """
        单个任务的前向传播（不更新参数）
        """
        # 移动到设备
        support_signals = support_signals.to(self.device).float().unsqueeze(1)
        support_labels = support_labels.to(self.device)
        query_signals = query_signals.to(self.device).float().unsqueeze(1)
        query_labels = query_labels.to(self.device)
        
        # 前向传播
        support_v, support_s = self.model(support_signals)
        query_v, query_s = self.model(query_signals)
        
        # 标签重映射
        unique_labels = torch.unique(support_labels)
        n_ways = len(unique_labels)
        label_mapping = {label.item(): i for i, label in enumerate(unique_labels)}
        
        remapped_support_labels = torch.tensor([label_mapping[label.item()] for label in support_labels], device=self.device)
        remapped_query_labels = torch.tensor([label_mapping[label.item()] for label in query_labels], device=self.device)
        
        # 计算高斯原型
        prototypes, precision_matrices = self.compute_gaussian_prototypes(
            support_v, support_s, remapped_support_labels, n_ways
        )
        
        # 计算损失
        loss, probabilities, distances = self.loss_fn(
            query_v, prototypes, precision_matrices, remapped_query_labels
        )
        
        # 计算准确率
        predictions = torch.argmax(probabilities, dim=1)
        accuracy = (predictions == remapped_query_labels).float().mean()
        
        return loss, accuracy.item()
    # Historical configuration omitted; provide values through private configuration.
    #     """
    #     元学习训练循环
    #     """
    #     self.model.train()
    # Historical configuration omitted; provide values through private configuration.

    #     print(f"开始元学习训练: {epochs} epochs, {num_tasks} tasks/epoch")
    #     for epoch in range(epochs):
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    #         # 显式地控制每个epoch的任务数量为 num_tasks
    #         with tqdm(total=num_tasks, unit="batch") as t_loader:
    #             for i, meta_batch in enumerate(train_loader):
    #                 if i >= num_tasks:
    #                     break  # 达到预定任务数后跳出循环

    #                 support_samples, support_labels, query_samples, query_labels = meta_batch
                    
    #                 # 批量训练，一次处理一个 meta-batch
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
                    
    #                 # 迭代 batch 中的每个任务
    # Historical configuration omitted; provide values through private configuration.
                        
                        
    #                     loss, acc = self.train_step(
    #                         support_samples[j], support_labels[j],
    #                         query_samples[j], query_labels[j],
    # Historical configuration omitted; provide values through private configuration.
    #                     )
    #                     meta_batch_loss += loss
    #                     meta_batch_acc += acc

    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    #                 total_loss += avg_loss
    #                 total_acc += avg_acc
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    #                 t_loader.update()  # 更新进度条

    #         scheduler.step()
    # Historical configuration omitted; provide values through private configuration.
    def train(self, train_loader, num_tasks=None, n_ways=None, epochs=None):
        """
        改进的元学习训练循环
        """
        num_tasks = _cfg_resolve('old/GPN_old.ipynb.cell0.GPNTrainer.train.num_tasks', num_tasks)
        n_ways = _cfg_resolve('old/GPN_old.ipynb.cell0.GPNTrainer.train.n_ways', n_ways)
        epochs = _cfg_resolve('old/GPN_old.ipynb.cell0.GPNTrainer.train.epochs', epochs)
        self.model.train()
        scheduler = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=_cfg_require('old/GPN_old.ipynb.cell0.GPNTrainer.train.step_size'), gamma=_cfg_require('old/GPN_old.ipynb.cell0.GPNTrainer.train.gamma'))
        
        print(f"开始元学习训练: {epochs} epochs, {num_tasks} tasks/epoch")
        
        for epoch in range(epochs):
            total_loss = 0
            total_acc = 0
            processed_tasks = 0
            
            # 使用DataLoader的自然批处理
            with tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}") as pbar:
                for batch_idx, meta_batch in enumerate(pbar):
                    if processed_tasks >= num_tasks:
                        break
                    
                    # 批处理训练步骤
                    loss, acc = self.train_step_batch(meta_batch)
                    
                    total_loss += loss
                    total_acc += acc
                    processed_tasks += len(meta_batch[0])  # 实际处理的任务数
                    
                    pbar.set_postfix(
                        loss=f'{loss:.4f}', 
                        accuracy=f'{acc:.4f}',
                        tasks=f'{processed_tasks}/{num_tasks}'
                    )
            
            scheduler.step()
            avg_loss = total_loss / len(train_loader)
            avg_acc = total_acc / len(train_loader)
            
            print(f"Epoch {epoch+1} - Loss: {avg_loss:.4f}, Accuracy: {avg_acc:.4f}")
    def evaluate(self, test_loader, n_ways):
        """
        评估模型性能
        """
        self.model.eval()
        total_acc = 0

        with torch.no_grad():
            for meta_task in tqdm(test_loader,desc="Evaluating"):
                support_signals, support_labels, query_signals, query_labels = meta_task

                # 移动到设备
                support_signals = support_signals.to(self.device).float().permute(1, 0, 2, 3)
                support_labels = support_labels.to(self.device).flatten()
                query_signals = query_signals.to(self.device).float().permute(1, 0, 2, 3)
                query_labels = query_labels.to(self.device).flatten()

                # 前向传播
                support_v, support_s = self.model(support_signals)

                # # 重新映射标签
                unique_labels = torch.unique(support_labels)
                label_mapping = {label.item(): i for i, label in enumerate(unique_labels)}
                
                remapped_support_labels = torch.tensor([label_mapping[label.item()] for label in support_labels], 
                                                    device=self.device)
                remapped_query_labels = torch.tensor([label_mapping[label.item()] for label in query_labels], 
                                                    device=self.device)

                # 计算高斯原型
                prototypes, precision_matrices = self.compute_gaussian_prototypes(
                    support_v, support_s, remapped_support_labels, n_ways
                )
                
                # 计算距离和准确率
                query_v, query_s = self.model(query_signals)
                distances = self.loss_fn.compute_mahalanobis_distance(
                    query_v, prototypes, precision_matrices
                )
                logits = -distances
                predictions = torch.argmax(logits, dim=1)
                accuracy = (predictions == remapped_query_labels).float().mean()
                total_acc += accuracy.item()
            
        avg_acc = total_acc / len(test_loader)
        print(f"\nFinal Test Accuracy: {avg_acc:.4f}")
        return avg_acc

def main():
    # 参数设置
    FEATURE_DIM = _cfg_require('old/GPN_old.ipynb.cell0.main.FEATURE_DIM')
    N_WAY = _cfg_require('old/GPN_old.ipynb.cell0.main.N_WAY')
    K_SHOT = _cfg_require('old/GPN_old.ipynb.cell0.main.K_SHOT')
    Q_QUERY = _cfg_require('old/GPN_old.ipynb.cell0.main.Q_QUERY')
    NUM_TASKS = _cfg_require('old/GPN_old.ipynb.cell0.main.NUM_TASKS')
    EPOCHS = _cfg_require('old/GPN_old.ipynb.cell0.main.EPOCHS')
    LR = _cfg_require('old/GPN_old.ipynb.cell0.main.LR')
    BATCH_SIZE = _cfg_require('old/GPN_old.ipynb.cell0.main.BATCH_SIZE')
    SEED = _cfg_require('old/GPN_old.ipynb.cell0.main.SEED')
    
    # 数据路径
    # current_path=Path(__file__).resolve().parent
    dataset_path = _cfg_require('old/GPN_old.ipynb.cell0.main.dataset_path')
    
    # 创建原始数据集
    train_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=True,train_ratio=_cfg_require('old/GPN_old.ipynb.cell0.main.train_ratio'),max_sample_count=_cfg_require('old/GPN_old.ipynb.cell0.main.max_sample_count'))
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,train_ratio=_cfg_require('old/GPN_old.ipynb.cell0.main.train_ratio__2'),max_sample_count=_cfg_require('old/GPN_old.ipynb.cell0.main.max_sample_count__2'))
    
    # 创建元任务划分集
    meta_train_dataset = MetaDataset(train_dataset, N_WAY, K_SHOT, Q_QUERY,batch_size=BATCH_SIZE,num_tasks=NUM_TASKS,epochs=EPOCHS)
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,batch_size=BATCH_SIZE,num_tasks=NUM_TASKS,epochs=_cfg_require('old/GPN_old.ipynb.cell0.main.epochs'))
    
    # DataLoader
    train_loader = DataLoader(meta_train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=_cfg_require('old/GPN_old.ipynb.cell0.main.num_workers'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('old/GPN_old.ipynb.cell0.main.batch_size'), shuffle=False)
    #创建模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model=GPN(feature_dim=FEATURE_DIM)

    # 创建训练器
    meta_learner = GPNTrainer(model,device=device,feature_dim=FEATURE_DIM,lr=LR)    
    print("开始训练...")
    meta_learner.train(train_loader=train_loader,num_tasks=NUM_TASKS,n_ways=N_WAY,epochs=EPOCHS)
    
    print("训练完成！")

    meta_learner.evaluate(test_loader=test_loader,n_ways=N_WAY)

if __name__ == '__main__':
    main()
